In [19]:
# 데이터 생성 : csv파일을 불러옴

import pandas as pd

df = pd.read_csv('../../data/csv/sentiment_data_long.csv')
df.head()

,sentence,label
0,음악 선곡이 가슴을 울리고 배우들의 몰입감이 엄청나며 연출력이 세계적인 수준이고 그...,0
1,연출력이 세계적인 수준이고 음악 선곡이 가슴을 울리고 배우들의 몰입감이 엄청나며 그...,0
2,초반부 전개가 매우 흥미진진해서 배우들의 몰입감이 엄청나며 영상미가 압도적으로 화려...,0
3,초반부 전개가 매우 흥미진진해서 영상미가 압도적으로 화려하고 연출력이 세계적인 수준...,0
4,영상미가 압도적으로 화려하고 연출력이 세계적인 수준이고 음악 선곡이 가슴을 울리고 ...,0


In [20]:
# 독립, 종속 변수 분리

X = df['sentence']
y = df['label']


In [21]:
# 훈련, 테스트 분리

DATA_SIZE = 1000
TRAIN_SIZE = int(DATA_SIZE * 0.8)

X_train, X_test = X[:TRAIN_SIZE], X[TRAIN_SIZE:]
y_train, y_test = y[:TRAIN_SIZE], y[TRAIN_SIZE:]

In [22]:
from konlpy.tag import Okt

def get_preprocessing(sentence):
  okt = Okt()
  result = okt.pos(sentence, stem=True) # 문장을 형태소별로 나눔. 단 원형으로 
  words = [word for word, pos in result]
  return " ".join(words)


print(X_train[0])
print(get_preprocessing(X_train[0]))

# X_train과 X_test의 있는 문장들을 get_preprocessing을 적용
X_train = X_train.apply(get_preprocessing)
X_test = X_test.apply(get_preprocessing)

음악 선곡이 가슴을 울리고 배우들의 몰입감이 엄청나며 연출력이 세계적인 수준이고 그러나 결국에는 실망스러운 완성도에 화가 날 정도다.
음악 선곡 이 가슴 을 울리다 배우 들 의 몰입 감 이 엄청나다 연출 력 이 세계 적 인 수준 이고 그러나 결국 에는 실망 스러운 완성 도 에 화가 날 정도 다 .


In [23]:
# 벡터화
from tensorflow.keras import layers, models
import tensorflow as tf

vectorize_layer = layers.TextVectorization(
  max_tokens = 1000,
  output_mode = "int",
	output_sequence_length=10
)
vectorize_layer.adapt(X_train.tolist())


In [24]:
import tensorflow as tf

# 모델 설계
model = models.Sequential([
  layers.Input((1,), dtype= tf.string),
  vectorize_layer,
  # 히든층
  layers.Embedding(input_dim = 1000, output_dim = 64),
	layers.LSTM(32),
  layers.Dense(1, activation='sigmoid') # 출력층 (결과가 0,1이니까 sigmoid)  
])

In [25]:
# 모델 설정
model.compile(
  optimizer = 'adam',
  loss = 'binary_crossentropy', 
  metrics=['accuracy']
)

In [26]:
# 학습
history = model.fit(
  X_train.values, y_train, epochs=100, verbose=1, validation_split=0.2, 
  batch_size=32
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.8516 - loss: 0.5548 - val_accuracy: 0.6375 - val_loss: 0.6625
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9906 - loss: 0.1488 - val_accuracy: 1.0000 - val_loss: 0.3082
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0543 - val_accuracy: 1.0000 - val_loss: 0.1722
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0313 - val_accuracy: 1.0000 - val_loss: 0.0930
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 0.0185 - val_accuracy: 1.0000 - val_loss: 0.0508
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 1.0000 - loss: 0.0121 - val_accuracy: 1.0000 - val_loss: 0.0323
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.0087 - val_accuracy: 1.0000 - val_loss: 0.0217
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0063 - val_accuracy: 1.000

In [27]:
# 예측

import numpy as np

sentences = [
	'음악 선곡이 가슴을 울리고 배우들의 몰입감이 엄청나며 연출력이 세계적인 수준이다.',
	'지루한 장면이 반복되어서 스토리가 너무 개연성이 없고 결말이 허무하기 짝이 없다'
]

# 학습 데이터가 긍정 -> 부정, 부정 -> 긍정인 데이터들이어서 긍정 문구가 있으면 부정이 될거라고 예측
# 부정 문구가 있으면 긍정으로 될거라고 예측 해버림..


# 머신러닝 / 단순 MLP 방식의 문장 분석
#		-> 각 단어들이 긍정에 많은지, 부정에 많은지를 계산해서 그 결과를 출력
#		-> 앞에 긍정적인 긴 내용이다가 뒤에 짧은 부정이면 긍정으로 판별
# RNN 방식의 문장 분석
# 	-> 문장의 순서와 단어 사이의 관계를 학습
#		-> 앞에 긍정적인 긴 내용이다가 뒤에 짧은 부정이면
#			 학습 내용에 따라 부정으로 판별할 수 있음 
#				-> 학습 데이터에 반전된 문장이 없으면 판별 못함
#				-> 학습 데이터에 반전된 문장이 없으면 판별 함


predictions = model.predict(np.array(sentences, dtype=object))
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step


array([[1.13869006e-04],
       [9.32921529e-01]], dtype=float32)